## Silver Layer
###### banking.bronze.cbs_customer -> banking.silver.dim_customer (masked, MERGE)

In [0]:
from pyspark.sql.functions import col, concat, lit, substring, upper, trim, to_date, current_timestamp
from delta.tables import DeltaTable

In [0]:
bronze_source="banking.bronze.cbs_customer"
silver_target= "banking.silver.dim_customer"
silver_table_name="dim_customer"

In [0]:
base="abfss://data@stnajdbanking.dfs.core.windows.net"
checkpoint= f"{base}/silver/{silver_table_name}/checkpoints/"
key= "customer_id"

#### Transformation
###### Cleanup step for customer records before they’re merged into the silver table.

In [0]:
def transform(df):
    return (
        df.dropDuplicates([key]) #de-duplicates rows by key, which is set earlier to customer_id
        # Normalize data types/formatting
        .withColumn("dob", to_date("dob")) 
        .withColumn("gender", upper(trim(col("gender"))))
        # PII masking (PDPL / SAMA): keep last 4 digits and initials only
        .withColumn("national_id_or_iqama",
                    concat(lit("******"), substring(col("national_id_or_iqama").cast("string"), -4, 4)))
        .withColumn("first_name", concat(substring("first_name", 1, 1), lit("***")))
        .withColumn("last_name",  concat(substring("last_name", 1, 1), lit("***")))
        # Add load_timestamp for auditability.
        .withColumn("load_timestamp", current_timestamp())
        # Drop ingestion-only metadata columns
        .drop("source_file", "bronze_ingested_at")
    )

### Loading into Silver
###### First batch creates the table, later batches keep it sync

In [0]:
# Create the empty target once, with the right schema. mode("ignore") does nothing if it already exists,
# so a later run can never overwrite the dimension.
transform(spark.read.table(bronze_source).limit(0)).write.format("delta").mode("ignore").saveAsTable(silver_target)

In [0]:
def upsert(batch_df, batch_id):
    #s = batch_df.sparkSession            # Takes one micro-batch of rows from the bronze stream
    batch_df = transform(batch_df)
    if not s.catalog.tableExists(silver_target):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_target)   # first run
    else:
        (DeltaTable.forName(s, silver_target).alias("t")
            .merge(batch_df.alias("s"), f"t.{key} = s.{key}") # Performs a Delta Lake merge on key (customer_id):
            .whenMatchedUpdateAll() # Matching keys are updated
            .whenNotMatchedInsertAll() # New keys are inserted
            .execute())


In [0]:
(
    spark.readStream.table(bronze_source)
    .writeStream
    .foreachBatch(upsert)
    .outputMode("update")
    .option("checkpointLocation", checkpoint) # Stores progress so the stream can resume safely and avoid reprocessing completed batches.
    .trigger(availableNow=True) # Process all currently available data, then stop
    .start()
    .awaitTermination()
)

In [0]:
display(spark.table(silver_target))